# Leakage Proxy Audit for the Paper

This notebook estimates **interviewer-language leakage proxies** across transcript versions.
It is intended as a paper-facing audit, not as a ground-truth measurement of leakage.

## Headline metrics

- **Strict heuristic leakage**: primary estimate.
- **Broad heuristic leakage**: upper-bound sensitivity check.
- **Temporal overlap with interviewer spans**: timing-based sanity check.

## Compared transcript versions

- `original_edaic_proxy`
- `raw_whisper_participant_speaker`
- `fixed_diarization_participant_only`

## Notes

- Raw Whisper participant speech is inferred by mapping the fixed `role_labeled` output back to the raw diarized speaker ids via `source_turn_idx`.
- The overlap metric uses fixed diarization interviewer spans as the reference, so the fixed participant-only version should go to zero by construction.
- For the paper, the most defensible headline number is the **corpus-weighted strict heuristic leakage by words**.


In [1]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
DATA_ROOT = REPO_ROOT.parent.parent
OUTPUT_DIR = DATA_ROOT / "tmp" / "leakage_audit_paper_2026-03-26"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORIG_DIR = DATA_ROOT / "datasets" / "old" / "woz_end_whisper_test"
RAW_DIR = DATA_ROOT / "datasets" / "raw_transcribes" / "12_03_26" / "raw_translated" / "transcribed"
FIXED_DIR = DATA_ROOT / "tmp" / "hybrid_cleanup_smoke" / "participant_only"
ROLE_DIR = DATA_ROOT / "tmp" / "hybrid_cleanup_smoke" / "role_labeled"

EN_QUESTION_PREFIXES = (
    "how ",
    "what ",
    "why ",
    "when ",
    "where ",
    "who ",
    "which ",
    "do you ",
    "did you ",
    "are you ",
    "have you ",
    "can you ",
    "could you ",
    "would you ",
    "tell me ",
)
UK_QUESTION_PREFIXES = (
    "як ",
    "що ",
    "чому ",
    "коли ",
    "де ",
    "хто ",
    "який ",
    "чи ",
    "розкажи ",
    "розкажіть ",
)

EN_SECOND_PERSON = (" you ", " your ", " you're ", " you've ", " yourself ")
UK_SECOND_PERSON = (" ти ", " тобі ", " тебе ", " ваш ", " вам ", " вас ")

HARD_PROMPT_PHRASES_EN = (
    "i'd love to hear all about it",
    "move around a little bit",
    "xbox kinect",
    "virtual human",
    "repeat that",
    "interview in spanish",
    "thanks for sharing your thoughts with me",
    "thank you for sharing your thoughts with me",
    "can you tell me more about that",
    "have a good day",
    "you have a good day",
    "goodbye",
    "you're welcome",
    "you are welcome",
    "that is so interesting",
    "i'm sorry",
    "i am sorry",
)
HARD_PROMPT_PHRASES_UK = (
    "віртуальною людиною",
    "рухайся трохи",
    "дякую, що поділилися своїми думками",
    "розкажи мені більше про це",
    "розкажіть мені більше про це",
    "гарного дня",
    "до побачення",
    "будь ласка",
    "мені шкода",
)

SOURCE_ORDER = [
    "original_edaic_proxy",
    "raw_whisper_participant_speaker",
    "fixed_diarization_participant_only",
]


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def segment_duration(seg: dict[str, Any]) -> float:
    return max(0.0, float(seg.get("end", 0.0)) - float(seg.get("start", 0.0)))


def segment_word_count(seg: dict[str, Any]) -> int:
    words = seg.get("words") or []
    if words:
        return len(words)
    text = normalize_text(seg.get("text", ""))
    return len([token for token in text.split(" ") if token])


def overlap(a0: float, a1: float, b0: float, b1: float) -> float:
    return max(0.0, min(a1, b1) - max(a0, b0))


def has_any(text_l: str, terms: tuple[str, ...]) -> bool:
    wrapped = f" {text_l} "
    return any(term in wrapped for term in terms)


def starts_question_prefix(text_l: str, lang_hint: str) -> bool:
    prefixes = UK_QUESTION_PREFIXES if lang_hint == "uk" else EN_QUESTION_PREFIXES
    return text_l.startswith(prefixes)


def has_second_person(text_l: str, lang_hint: str) -> bool:
    terms = UK_SECOND_PERSON if lang_hint == "uk" else EN_SECOND_PERSON
    return has_any(text_l, terms)


def starts_imperative_prompt(text_l: str) -> bool:
    return text_l.startswith((
        "spell ",
        "repeat ",
        "move around",
        "just move around",
        "let me know",
        "tell me ",
    ))


def heuristic_reasons(text: str, lang_hint: str = "en") -> tuple[str | None, str | None]:
    text_n = normalize_text(text)
    if not text_n:
        return None, None

    text_l = text_n.lower()
    word_count = len([token for token in text_n.split(" ") if token])
    hard_phrases = HARD_PROMPT_PHRASES_UK if lang_hint == "uk" else HARD_PROMPT_PHRASES_EN

    strict_reasons: list[str] = []
    broad_reasons: list[str] = []

    if any(phrase in text_l for phrase in hard_phrases):
        strict_reasons.append("hard_phrase")
        broad_reasons.append("hard_phrase")

    if starts_imperative_prompt(text_l):
        strict_reasons.append("imperative_prompt")
        broad_reasons.append("starts_imperative")

    if "?" in text_n:
        strict_reasons.append("question_mark")

    if starts_question_prefix(text_l, lang_hint) and has_second_person(text_l, lang_hint):
        strict_reasons.append("qprefix+2p")

    if starts_question_prefix(text_l, lang_hint):
        broad_reasons.append("starts_qprefix")

    if word_count <= 4 and (
        starts_question_prefix(text_l, lang_hint)
        or text_l.startswith(("from what", "and what", "for what"))
    ):
        broad_reasons.append("short_qword")

    if word_count <= 6 and has_second_person(text_l, lang_hint):
        broad_reasons.append("short_2p")

    strict = "|".join(dict.fromkeys(strict_reasons)) or None
    broad = "|".join(dict.fromkeys(broad_reasons)) or None
    return strict, broad


def build_raw_speaker_map(file_id: str) -> dict[str, Any]:
    raw_segments = load_json(RAW_DIR / f"{file_id}.json")["segments"]
    role_segments = load_json(ROLE_DIR / f"{file_id}.json")["segments"]

    role_counts: dict[str, Counter[str]] = defaultdict(Counter)
    for seg in role_segments:
        source_turn_idx = seg.get("source_turn_idx")
        if source_turn_idx is None or source_turn_idx >= len(raw_segments):
            continue
        speaker = raw_segments[source_turn_idx].get("speaker", "")
        role = seg.get("role", "unknown")
        role_counts[speaker][role] += segment_word_count(seg)

    participant_speaker = max(role_counts, key=lambda speaker: role_counts[speaker]["participant"])
    return {
        "participant_speaker": participant_speaker,
        "role_word_counts": {speaker: dict(counts) for speaker, counts in role_counts.items()},
    }


def participant_raw_segments(file_id: str, participant_speaker_by_file: dict[str, str]) -> list[dict[str, Any]]:
    raw_segments = load_json(RAW_DIR / f"{file_id}.json")["segments"]
    participant_speaker = participant_speaker_by_file[file_id]
    return [
        seg
        for seg in raw_segments
        if seg.get("speaker") == participant_speaker and normalize_text(seg.get("text", ""))
    ]


def overlap_breakdown(file_id: str, source: str, segments: list[dict[str, Any]]) -> dict[str, Any]:
    ref_segments = [
        seg
        for seg in load_json(ROLE_DIR / f"{file_id}.json")["segments"]
        if normalize_text(seg.get("text", ""))
    ]
    total_overlap = 0.0
    interviewer_overlap = 0.0
    unknown_overlap = 0.0

    for seg in segments:
        if not normalize_text(seg.get("text", "")):
            continue
        seg_start = float(seg.get("start", 0.0))
        seg_end = float(seg.get("end", 0.0))
        if seg_end <= seg_start:
            continue
        for ref in ref_segments:
            dt = overlap(seg_start, seg_end, float(ref.get("start", 0.0)), float(ref.get("end", 0.0)))
            if dt <= 0:
                continue
            total_overlap += dt
            if ref.get("role") == "interviewer":
                interviewer_overlap += dt
            elif ref.get("role") == "unknown":
                unknown_overlap += dt

    return {
        "file": file_id,
        "source": source,
        "overlap_total_sec": total_overlap,
        "overlap_interviewer_sec": interviewer_overlap,
        "overlap_unknown_sec": unknown_overlap,
        "overlap_pct": float(interviewer_overlap / total_overlap * 100.0) if total_overlap else np.nan,
    }


def audit_source_segments(file_id: str, source: str, segments: list[dict[str, Any]]) -> tuple[dict[str, Any] | None, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    total_words = 0
    total_duration = 0.0
    strict_flagged_words = 0
    broad_flagged_words = 0
    strict_flagged_duration = 0.0
    broad_flagged_duration = 0.0

    for segment_idx, seg in enumerate(segments):
        text = normalize_text(seg.get("text", ""))
        if not text:
            continue

        strict_reason, broad_reason = heuristic_reasons(text, lang_hint="en")
        words = max(1, segment_word_count(seg))
        duration = max(0.01, segment_duration(seg))

        total_words += words
        total_duration += duration
        if strict_reason:
            strict_flagged_words += words
            strict_flagged_duration += duration
        if strict_reason or broad_reason:
            broad_flagged_words += words
            broad_flagged_duration += duration

        rows.append(
            {
                "file": file_id,
                "source": source,
                "segment_idx": segment_idx,
                "start": float(seg.get("start", 0.0)),
                "end": float(seg.get("end", 0.0)),
                "words": words,
                "duration_sec": duration,
                "text": text,
                "strict_reason": strict_reason,
                "broad_reason": broad_reason,
                "flagged_strict": bool(strict_reason),
                "flagged_broad": bool(strict_reason or broad_reason),
            }
        )

    detail_df = pd.DataFrame(rows)
    if detail_df.empty:
        return None, detail_df

    summary = {
        "file": file_id,
        "source": source,
        "segments": int(len(detail_df)),
        "words": float(total_words),
        "duration_sec": float(total_duration),
        "strict_flagged_segments": int(detail_df["flagged_strict"].sum()),
        "broad_flagged_segments": int(detail_df["flagged_broad"].sum()),
        "strict_flagged_words": float(strict_flagged_words),
        "broad_flagged_words": float(broad_flagged_words),
        "strict_flagged_duration": float(strict_flagged_duration),
        "broad_flagged_duration": float(broad_flagged_duration),
        "strict_leakage_pct_words": float(strict_flagged_words / total_words * 100.0),
        "broad_leakage_pct_words": float(broad_flagged_words / total_words * 100.0),
        "strict_leakage_pct_duration": float(strict_flagged_duration / total_duration * 100.0),
        "broad_leakage_pct_duration": float(broad_flagged_duration / total_duration * 100.0),
    }
    return summary, detail_df


def run_analysis() -> dict[str, pd.DataFrame]:
    common_ids = sorted(
        {p.stem for p in ORIG_DIR.glob("*.json")}
        & {p.stem for p in RAW_DIR.glob("*.json")}
        & {p.stem for p in FIXED_DIR.glob("*.json")}
        & {p.stem for p in ROLE_DIR.glob("*.json")}
    )

    raw_speaker_rows: list[dict[str, Any]] = []
    participant_speaker_by_file: dict[str, str] = {}
    for file_id in common_ids:
        speaker_map = build_raw_speaker_map(file_id)
        participant_speaker_by_file[file_id] = speaker_map["participant_speaker"]
        for speaker, counts in speaker_map["role_word_counts"].items():
            raw_speaker_rows.append(
                {
                    "file": file_id,
                    "speaker": speaker,
                    "participant_words": counts.get("participant", 0),
                    "interviewer_words": counts.get("interviewer", 0),
                    "unknown_words": counts.get("unknown", 0),
                    "selected_as_participant": speaker == speaker_map["participant_speaker"],
                }
            )
    raw_speaker_map_df = pd.DataFrame(raw_speaker_rows).sort_values(["file", "speaker"]).reset_index(drop=True)

    summary_rows: list[dict[str, Any]] = []
    detail_frames: list[pd.DataFrame] = []
    overlap_rows: list[dict[str, Any]] = []
    for file_id in common_ids:
        source_payloads = [
            ("original_edaic_proxy", load_json(ORIG_DIR / f"{file_id}.json")["segments"]),
            ("raw_whisper_participant_speaker", participant_raw_segments(file_id, participant_speaker_by_file)),
            ("fixed_diarization_participant_only", load_json(FIXED_DIR / f"{file_id}.json")["segments"]),
        ]
        for source, segments in source_payloads:
            summary, detail = audit_source_segments(file_id, source, segments)
            if summary is None:
                continue
            summary_rows.append(summary)
            detail_frames.append(detail)
            overlap_rows.append(overlap_breakdown(file_id, source, segments))

    per_file_scores = pd.DataFrame(summary_rows).sort_values(["source", "file"]).reset_index(drop=True)
    per_segment_scores = pd.concat(detail_frames, ignore_index=True)
    overlap_scores = pd.DataFrame(overlap_rows).sort_values(["source", "file"]).reset_index(drop=True)

    corpus_summary = (
        per_file_scores.groupby("source", sort=False)[
            [
                "segments",
                "words",
                "duration_sec",
                "strict_flagged_segments",
                "broad_flagged_segments",
                "strict_flagged_words",
                "broad_flagged_words",
                "strict_flagged_duration",
                "broad_flagged_duration",
            ]
        ]
        .sum()
        .reindex(SOURCE_ORDER)
    )
    overlap_totals = (
        overlap_scores.groupby("source", sort=False)[
            ["overlap_total_sec", "overlap_interviewer_sec", "overlap_unknown_sec"]
        ]
        .sum()
        .reindex(SOURCE_ORDER)
    )
    corpus_summary = corpus_summary.join(overlap_totals)
    corpus_summary["strict_leakage_pct_words"] = corpus_summary["strict_flagged_words"] / corpus_summary["words"] * 100.0
    corpus_summary["broad_leakage_pct_words"] = corpus_summary["broad_flagged_words"] / corpus_summary["words"] * 100.0
    corpus_summary["strict_leakage_pct_duration"] = corpus_summary["strict_flagged_duration"] / corpus_summary["duration_sec"] * 100.0
    corpus_summary["broad_leakage_pct_duration"] = corpus_summary["broad_flagged_duration"] / corpus_summary["duration_sec"] * 100.0
    corpus_summary["overlap_pct"] = corpus_summary["overlap_interviewer_sec"] / corpus_summary["overlap_total_sec"] * 100.0
    corpus_summary["unknown_overlap_pct"] = corpus_summary["overlap_unknown_sec"] / corpus_summary["overlap_total_sec"] * 100.0
    corpus_summary = corpus_summary.reset_index().rename(columns={"index": "source"})

    paper_table = corpus_summary[
        [
            "source",
            "strict_leakage_pct_words",
            "broad_leakage_pct_words",
            "strict_leakage_pct_duration",
            "broad_leakage_pct_duration",
            "overlap_pct",
        ]
    ].round(3)

    per_file_summary = (
        per_file_scores.groupby("source", sort=False)[
            [
                "strict_leakage_pct_words",
                "broad_leakage_pct_words",
                "strict_leakage_pct_duration",
                "broad_leakage_pct_duration",
            ]
        ]
        .agg(["mean", "median"])
        .reindex(SOURCE_ORDER)
        .round(3)
    )

    flagged_segments = per_segment_scores[per_segment_scores["flagged_broad"]].copy()
    fixed_examples = (
        flagged_segments[flagged_segments["source"] == "fixed_diarization_participant_only"]
        .assign(reason=lambda df: df["strict_reason"].fillna(df["broad_reason"]))
        .sort_values(["flagged_strict", "words", "duration_sec"], ascending=[False, False, False])
        [["file", "segment_idx", "words", "duration_sec", "strict_reason", "broad_reason", "text"]]
        .reset_index(drop=True)
    )

    paper_table.to_csv(OUTPUT_DIR / "paper_table.csv", index=False)
    corpus_summary.round(3).to_csv(OUTPUT_DIR / "corpus_summary.csv", index=False)
    per_file_summary.to_csv(OUTPUT_DIR / "per_file_summary.csv")
    per_file_scores.to_csv(OUTPUT_DIR / "per_file_scores.csv", index=False)
    overlap_scores.to_csv(OUTPUT_DIR / "overlap_scores.csv", index=False)
    flagged_segments.to_csv(OUTPUT_DIR / "flagged_segments.csv", index=False)
    fixed_examples.to_csv(OUTPUT_DIR / "fixed_examples.csv", index=False)
    raw_speaker_map_df.to_csv(OUTPUT_DIR / "raw_speaker_mapping.csv", index=False)

    return {
        "common_ids": pd.DataFrame({"matched_interviews": [len(common_ids)]}),
        "paper_table": paper_table,
        "corpus_summary": corpus_summary.round(3),
        "per_file_summary": per_file_summary,
        "per_file_scores": per_file_scores,
        "overlap_scores": overlap_scores,
        "flagged_segments": flagged_segments,
        "fixed_examples": fixed_examples,
        "raw_speaker_mapping": raw_speaker_map_df,
        "output_dir": pd.DataFrame({"path": [str(OUTPUT_DIR)]}),
    }


/var/folders/1x/xhy9y57n2xz37yb7t8t2rdfr0000gn/T/ipykernel_93766/4230495299.py:10: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## Interpretation

For the paper, use the following framing.

- **Primary estimate**: `strict_leakage_pct_words`
- **Upper bound / sensitivity check**: `broad_leakage_pct_words`
- **Timing sanity check**: `overlap_pct`

Residual fixed-diarization hits should be inspected as likely false positives before being interpreted as true interviewer carryover.


In [2]:
results = run_analysis()

print(f"Matched interviews: {int(results['common_ids'].iloc[0, 0])}")
print(f"Saved outputs to: {results['output_dir'].iloc[0, 0]}")
print()
print("Paper table (corpus-weighted):")
display(results["paper_table"])
print()
print("Per-file mean / median summary:")
display(results["per_file_summary"])
print()
print("Example flagged segments from fixed_diarization_participant_only:")
display(results["fixed_examples"].head(25))


Matched interviews: 136
Saved outputs to: /Users/poluidol/Documents/work/airest/airest_perp_research/data_analysis/tmp/leakage_audit_paper_2026-03-26

Paper table (corpus-weighted):


,source,strict_leakage_pct_words,broad_leakage_pct_words,strict_leakage_pct_duration,broad_leakage_pct_duration,overlap_pct
0,original_edaic_proxy,9.743,15.586,5.011,9.242,9.222
1,raw_whisper_participant_speaker,10.737,12.107,9.587,10.857,9.032
2,fixed_diarization_participant_only,2.099,3.500,2.058,3.433,0.000



Per-file mean / median summary:


strict_leakage_pct_words          \
                                                       mean  median   
source                                                                
original_edaic_proxy                                  9.978   9.326   
raw_whisper_participant_speaker                      14.217  11.409   
fixed_diarization_participant_only                    2.035   1.189   

                                   broad_leakage_pct_words          \
                                                      mean  median   
source                                                               
original_edaic_proxy                                15.759  15.653   
raw_whisper_participant_speaker                     15.617  13.202   
fixed_diarization_participant_only                   3.452   2.840   

                                   strict_leakage_pct_duration         \
                                                          mean median   
source                                                                  
original_edaic_proxy                                     6.520  5.022   
raw_whisper_participant_speaker                         11.977  9.362   
fixed_diarization_participant_only                       2.018  0.876   

                                   broad_leakage_pct_duration          
                                                         mean  median  
source                                                                 
original_edaic_proxy                                   10.194   8.368  
raw_whisper_participant_speaker                        13.201  10.198  
fixed_diarization_participant_only                      3.432   2.386


Example flagged segments from fixed_diarization_participant_only:


,file,segment_idx,words,duration_sec,strict_reason,broad_reason,text
0,636,66,74,25.535,qprefix+2p,starts_qprefix,when i was younger my dream job was to be an a...
1,608,40,66,26.480,hard_phrase,hard_phrase,um say again i'm sorry i don't really understa...
2,627,85,61,29.045,qprefix+2p,starts_qprefix,when i don't sleep well it's pretty clear i'm ...
3,633,119,53,29.214,hard_phrase,hard_phrase,um there's so many of those i wish i would hav...
4,414,82,43,16.447,question_mark,None,I guess just sometimes they reflect things tha...
5,632,22,40,27.375,hard_phrase,hard_phrase,although it was kind of a bittersweet goodbye ...
6,718,153,40,11.861,qprefix+2p,starts_qprefix,what i'm most proud of uh i can say one of the...
7,420,57,39,14.135,qprefix+2p,starts_qprefix,"When somebody talks to you, when they say some..."
8,677,105,38,29.348,hard_phrase,hard_phrase,see getting away from drugs and alcohol pretty...
9,430,76,34,21.755,hard_phrase,hard_phrase,I'm sorry say again gotcha it does I think it ...
